In [ ]:
import os
MY_KEY = os.environ.get("MY_KEY")

In [26]:
import json
import time
from openai import OpenAI
from pathlib import Path
from tqdm import tqdm

# Set OpenAI API key
client = OpenAI(api_key=MY_KEY)

# Load Q&A pairs
input_path = Path("../data/medquad_gen_mcqs/medquad_sampled.jsonl")
output_path = Path("../data/generated_mcqs_gpt.jsonl")

with open(input_path, "r") as f:
    qa_data = [json.loads(line) for line in f]

# Prompt formatting
def format_prompt(question, answer):
    return f"""Convert the following medical Q&A into a multiple-choice question.

Question: {question}
Answer: {answer}

Provide output in this XML format:

<question>...</question>
<choices>
A. ...
B. ...
C. ...
D. ...
</choices>
<reason>...</reason>
<answer>C</answer>
"""

# Retry-enabled GPT call
def generate_mcq(prompt, model="gpt-3.5-turbo", retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=512
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(1)
            else:
                print(f"Failed after {retries} attempts: {e}")
                return None

# Main generation loop
if output_path.exists():
    response = input(f"File {output_path} exists. Delete it? (y/n): ")
    if response.lower().startswith("y"):
        output_path.unlink()
    else:
        raise RuntimeError("Aborting to avoid overwriting the file.")

with open(output_path, "a") as f:
    for idx, row in tqdm(enumerate(qa_data), total=len(qa_data), desc="Generating MCQs"):
        prompt = format_prompt(row["question"], row["answer"])
        completion = generate_mcq(prompt)
        if completion:
            entry = {
                "question": row["question"],
                "answer": row["answer"],
                "completion": completion
            }
            f.write(json.dumps(entry) + "\n")
            f.flush()

print(f"Saved to {output_path}")


Generating MCQs: 100%|██████████| 3000/3000 [1:11:35<00:00,  1.43s/it]

Saved to ../data/generated_mcqs_gpt.jsonl
